# Mutual Fund Performance Analysis
This notebook computes key performance metrics for the 40 mutual fund schemes and visualizes results.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, plotly.express as px, plotly.graph_objects as go
from scipy import stats
%matplotlib inline


In [ ]:
# Load NAV history (daily)
nav_df = pd.read_csv('../../data/02_nav_history.csv', parse_dates=['date'])
nav_df.head()


In [ ]:
# Pivot to wide format: one column per scheme
nav_wide = nav_df.pivot(index='date', columns='scheme_id', values='nav').sort_index()
nav_wide.head()


In [ ]:
# Daily returns for each scheme
daily_ret = nav_wide.pct_change().dropna(how='all')
daily_ret.head()


In [ ]:
# Validate return distribution (histogram for a random fund)
sample_fund = daily_ret.columns[0]
fig = px.histogram(daily_ret[sample_fund], nbins=100, title=f'Daily Return Distribution - Fund {sample_fund}')
fig.show()


In [ ]:
# CAGR calculations (1,3,5 year)
def compute_cagr(series, years):
    start = series.iloc[0]
    end = series.iloc[-1]
    return (end/start) ** (1/years) - 1

cagr_dict = {}
for fund in nav_wide.columns:
    ser = nav_wide[fund].dropna()
    years_total = (ser.index[-1] - ser.index[0]).days/365.25
    cagr_1 = compute_cagr(ser[-int(365.25):], 1) if years_total>=1 else np.nan
    cagr_3 = compute_cagr(ser[-int(365.25*3):], 3) if years_total>=3 else np.nan
    cagr_5 = compute_cagr(ser[-int(365.25*5):], 5) if years_total>=5 else np.nan
    cagr_dict[fund] = {'1Y':cagr_1,'3Y':cagr_3,'5Y':cagr_5}
cagr_df = pd.DataFrame.from_dict(cagr_dict, orient='index')
cagr_df.head()


In [ ]:
# Sharpe Ratio (annualized)
rf_annual = 0.065
mean_ann = daily_ret.mean()*252
std_ann = daily_ret.std()*np.sqrt(252)
sharpe = (mean_ann - rf_annual)/std_ann
sharpe.name = 'Sharpe'
sharpe.head()


In [ ]:
# Sortino Ratio (downside risk)
downside = daily_ret.copy()
downside[downside>0] = 0
downside_std = np.sqrt((downside**2).mean()) * np.sqrt(252)
sortino = (mean_ann - rf_annual)/downside_std
sortino.name='Sortino'
sortino.head()


In [ ]:
# Load benchmark indices (Nifty 100)
bench = pd.read_csv('../../data/10_benchmark_indices.csv', parse_dates=['date'])
nifty100 = bench[bench['index']=='NIFTY100'].set_index('date')['close'].sort_index()
nifty100 = nifty100.reindex(nav_wide.index).ffill()
# Align returns
bench_ret = nifty100.pct_change().dropna()
# Alpha & Beta via OLS for each fund
alpha_beta = {}
for fund in daily_ret.columns:
    y = daily_ret[fund].loc[bench_ret.index]
    x = bench_ret
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    alpha = intercept * 252
    beta = slope
    alpha_beta[fund] = {'Alpha':alpha,'Beta':beta}
alpha_beta_df = pd.DataFrame.from_dict(alpha_beta, orient='index')
alpha_beta_df.head()


In [ ]:
# Maximum Drawdown per fund
def max_drawdown(series):
    roll_max = series.cummax()
    drawdown = series/roll_max - 1
    return drawdown.min()

dd = {fund:max_drawdown(nav_wide[fund].dropna()) for fund in nav_wide.columns}
dd_df = pd.Series(dd, name='MaxDrawdown')
dd_df.head()


In [ ]:
# Expense Ratio (assume column in aum_by_fund_house.csv)
exp = pd.read_csv('../../data/03_aum_by_fund_house.csv')
# Map fund to expense ratio – placeholder logic (use first matching row)
exp_ratio = exp.set_index('scheme_id')['expense_ratio']
exp_ratio = exp_ratio.reindex(nav_wide.columns)

# Rankings (higher is better)
rank_3yr = cagr_df['3Y'].rank(ascending=False)
rank_sharpe = sharpe.rank(ascending=False)
rank_alpha = alpha_beta_df['Alpha'].rank(ascending=False)
rank_exp = (-exp_ratio).rank(ascending=False)  # lower expense gets higher rank
rank_dd = (-dd_df).rank(ascending=False)

score = (0.30*rank_3yr + 0.25*rank_sharpe + 0.20*rank_alpha + 0.15*rank_exp + 0.10*rank_dd)
scorecard = pd.DataFrame({
    '3Y_Rank':rank_3yr, 'Sharpe_Rank':rank_sharpe, 'Alpha_Rank':rank_alpha,
    'Exp_Rank':rank_exp, 'DD_Rank':rank_dd, 'Score':score
})
scorecard = scorecard.sort_values('Score', ascending=False)
scorecard.head()


In [ ]:
# Benchmark comparison chart for top 5 funds vs Nifty 50 & Nifty 100 (3‑year window)
top5 = scorecard.head(5).index
start_date = nav_wide.index.max() - pd.DateOffset(years=3)
mask = (nav_wide.index >= start_date)
fig = go.Figure()
for fund in top5:
    fig.add_trace(go.Scatter(x=nav_wide.index[mask], y=nav_wide[fund][mask], name=f'Fund {fund}'))
# Nifty 50 (assume present in benchmark file)
nifty50 = bench[bench['index']=='NIFTY50'].set_index('date')['close'].reindex(nav_wide.index).ffill()
fig.add_trace(go.Scatter(x=nifty50.index[mask], y=nifty50[mask], name='Nifty 50', line=dict(dash='dash')) )
fig.add_trace(go.Scatter(x=nifty100.index[mask], y=nifty100[mask], name='Nifty 100', line=dict(dash='dot')) )
fig.update_layout(title='Top 5 Funds vs Benchmarks (3‑yr)', xaxis_title='Date', yaxis_title='NAV')
fig.show()
# Tracking error for each top fund
tracking_error = {}
for fund in top5:
    err = (daily_ret[fund].loc[bench_ret.index] - bench_ret).std()*np.sqrt(252)
    tracking_error[fund]=err
te_df = pd.Series(tracking_error, name='TrackingError')
te_df.head()
